# 1_2 Geração de POS Tags

Este notebook realiza o **Part-of-Speech Tagging (POS)** do corpus de notícias brasileiras usando um tagger baseado em regras linguísticas para o Português Brasileiro.

In [ ]:
import re
from collections import Counter
import pandas as pd
print('Bibliotecas importadas!')

Bibliotecas importadas!


## 1. Dicionário POS e Nomes Próprios

In [ ]:
POS_DICT = {'o':'ART','a':'ART','um':'ART','uma':'ART','de':'PREP','do':'PREP','da':'PREP','em':'PREP','no':'PREP','na':'PREP','para':'PREP','por':'PREP','com':'PREP','ao':'PREP','à':'PREP','e':'CONJ','ou':'CONJ','mas':'CONJ','que':'CONJ','se':'CONJ','eu':'PRON','ele':'PRON','ela':'PRON','você':'PRON','anunciou':'V','aprovou':'V','informou':'V','assinou':'V','registrou':'V','inaugurou':'V','fechou':'V','lançou':'V','divulgou':'V','apresentou':'V','alertou':'V','atingiu':'V','decidiu':'V','incluindo':'V','será':'V','hoje':'ADV','não':'ADV','também':'ADV','já':'ADV','mais':'ADV','muito':'ADV'}
NOMES_PROPRIOS = {'Lula','Silva','Luiz','Inácio','Fernando','Haddad','Tarcísio','Freitas','Roberto','Campos','Neto','Ricardo','Nunes','Arthur','Lira','Marina','Brasil','Brasília','Paulo','Rio','Janeiro','Nordeste','Belém','Pará','Petrobras','Vale','Embraer','Ambev','Totvs','B3','Sabesp','Fiocruz'}
print(f'POS Dict: {len(POS_DICT)} entradas | Nomes Próprios: {len(NOMES_PROPRIOS)} entradas')

POS Dict: 77 entradas | Nomes Próprios: 40 entradas


## 2. Função POS Tagger

In [ ]:
def pos_tag_pt(sentence):
    tokens = re.findall(r'[\w]+|[.,!?;]', sentence)
    tagged = []
    for token in tokens:
        tl = token.lower()
        if re.match(r'[.,!?;:]', token):         pos = 'PUNCT'
        elif token in NOMES_PROPRIOS:             pos = 'PROPN'
        elif tl in POS_DICT:                     pos = POS_DICT[tl]
        elif re.match(r'^\d+[\.,]?\d*%?$', token): pos = 'NUM'
        elif token.istitle() and len(token) > 2: pos = 'PROPN'
        elif tl.endswith(('ção','ões','mento')):  pos = 'SUBST'
        elif tl.endswith(('ado','ido','ando','endo','indo')): pos = 'V'
        elif tl.endswith(('al','ais','el')):      pos = 'ADJ'
        elif tl.endswith('mente'):               pos = 'ADV'
        else:                                    pos = 'SUBST'
        tagged.append((token, pos))
    return tagged
print('Função pos_tag_pt definida!')

Função pos_tag_pt definida!


## 3. Exemplo de POS Tagging

In [ ]:
sentenca = documentos[0]
resultado = pos_tag_pt(sentenca)
print(f'Sentença: {sentenca}\n')
print(f"{'Token':<25} {'POS'}")
print('-'*40)
for token, pos in resultado:
    print(f'{token:<25} {pos}')

Sentença: O presidente Luiz Inácio Lula da Silva assinou hoje em Brasília um pacote de medidas para estimular a economia brasileira, incluindo investimentos em infraestrutura e geração de empregos.

Token                     POS
----------------------------------------
O                         ART
presidente                SUBST
Luiz                      PROPN
Inácio                    PROPN
Lula                      PROPN
da                        PREP
Silva                     PROPN
assinou                   V
hoje                      ADV
em                        PREP
Brasília                  PROPN
um                        ART
pacote                    SUBST
de                        PREP
medidas                   SUBST
para                      PREP
estimular                 SUBST
a                         ART
economia                  SUBST
brasileira                SUBST
,                         PUNCT
incluindo                 V
investimentos             SUBST
em            

## 4. Distribuição de POS no Corpus

In [ ]:
todos_pos = []
for doc in documentos:
    tagged = pos_tag_pt(doc)
    todos_pos.extend(t[1] for t in tagged)
freq_pos = Counter(todos_pos)
print('Distribuição de POS no corpus:')
print('-'*30)
for pos, freq in freq_pos.most_common():
    print(f'{pos:<10} {freq:>5} ocorrências')

Distribuição de POS no corpus:
------------------------------
SUBST        219 ocorrências
PREP         151 ocorrências
PROPN        127 ocorrências
ART           75 ocorrências
PUNCT         69 ocorrências
V             54 ocorrências
NUM           25 ocorrências
CONJ          19 ocorrências
ADJ           18 ocorrências
ADV            4 ocorrências


In [ ]:
df_pos = pd.DataFrame([{'id':i+1,'documento':doc,'pos_tags':str(pos_tag_pt(doc))} for i,doc in enumerate(documentos)])
df_pos.to_csv('dataset_pos.csv', sep=';', index=False, encoding='utf-8')
print(f'dataset_pos.csv salvo! {len(df_pos)} documentos taggeados.')

dataset_pos.csv salvo! 32 documentos taggeados.
